**Navigation** : [Index](README.md) | [<< Précédent](10_LocalLlama.ipynb) | [Suivant >>](11_Quantization.ipynb)

# 10e. LLamaSharp : bake-off binding .NET de llama.cpp

**Durée estimée** : 50 minutes
**Prérequis** : C# asynchrone, base de llama.cpp, runtime .NET 8 self-contained
**Matériel du run de référence** : GPU NVIDIA RTX 3080 Ti 16 Go ; modèle GGUF Qwen3-4B Q4_K_M (~2.5 GB)

## Objectifs d'apprentissage

1. Brancher un binding .NET de `llama.cpp` (LLamaSharp 0.27.0) sur le même GPU que la Phase 1 (TensorSharp, PR #12645).
2. Charger un GGUF et mesurer `tok/s` (débit), VRAM, et taux de jetons `<pad>` — sans recopier un benchmark tiers.
3. Comparer à TensorSharp sur les mêmes invites et la même quantification Q4, et statuer sur l'utilité des deux moteurs pour le curriculum.
4. Diagnostiquer, puis **réparer**, une fausse détection de backend : pourquoi LLamaSharp 0.27 ignore une carte CUDA parfaitement fonctionnelle, et comment le corriger (règle F : réparer l'environnement, jamais le contourner).

## Contexte

Ce notebook est la **Phase 2** du bake-off [#12353](https://github.com/jsboige/CoursIA/issues/12353) qui compare trois moteurs d'inférence locale en .NET :

- **TensorSharp** (Phase 1, [#12645](https://github.com/jsboige/CoursIA/pull/12645)) : serveur HTTP distant sur RTX 3080, mais génération Gemma 4 contenant 159/160 jetons `<pad>` (`RECOVERABLE-LOCAL`).
- **LLamaSharp** (Phase 2, ce notebook) : binding .NET natif de `llama.cpp` 0.27.0 — load in-process via P/Invoke, pas de serveur distant.
- **ORT GenAI** (Phase 3, à venir) : ONNX Runtime GenAI Microsoft.

**Scope partitionné par le coordinateur (ai-01) le 2026-08-24** : `MyIA.AI.Notebooks/GenAI/Texte/23_*.ipynb` + `Texte/README.md` (lane `myia-po-2026:CoursIA-2`, claim posé sur l'issue #12353).

> **Verdict du pilote : SOTA-OK.** LLamaSharp charge Qwen3-4B Q4_K_M et le décode
> **sur la RTX 3080 Ti** — 37/37 couches sur `CUDA0`, deux backends enregistrés, VRAM en
> hausse de 3163 MiB pendant l'inférence — pour un débit agrégé de **34.62 tok/s** sur
> les quatre invites Phase 1, avec 0 jeton `<pad>`.
>
> Ce chiffre n'a pas toujours été celui-là, et l'histoire de sa correction est le cœur
> pédagogique de ce notebook. Le pilote a d'abord mesuré **14.14 tok/s** en croyant
> mesurer un GPU : le backend CUDA ne s'était jamais enregistré et les 353 tokens
> tombaient sur le CPU, **sans erreur ni avertissement**. Le débit était authentique, son
> *attribution* était fausse. La cause a été remontée jusqu'au bout puis **réparée**
> (§1bis) — d'où le facteur **×2.45** entre les deux régimes, mesuré à invites, modèle et
> échantillonnage identiques (`Temperature = 0.0` ⇒ exactement 353 tokens des deux côtés).
>
> Deux leçons transposables : **un paramètre demandé n'est pas une mesure**
> (`GpuLayerCount = 99` ne prouve rien), et **un backend qui ne se charge pas retombe en
> silence** sur le CPU. Le §1ter donne les observables qui tranchent.


In [1]:
import platform, subprocess, json, os

print("=" * 70)
print("Cellule 1 — Pré-flight environnement")
print("=" * 70)
print(f"OS             : {platform.platform()}")
print(f"Architecture   : {platform.machine()}")
print(f"Python         : {platform.python_version()}")

# dotnet SDK + runtimes via --list-runtimes
r = subprocess.run(["dotnet", "--list-runtimes"], capture_output=True, text=True, shell=True)
print("\nRuntimes .NET visibles :")
for line in r.stdout.strip().splitlines():
    print(f"  {line}")
print("\n→ Le runtime .NET 8.0.27 (C:\\dotnet-manual) est requis pour LLamaSharp 0.27.0 (marké compatible net8.0 par NuGet).")


Cellule 1 — Pré-flight environnement
OS             : Windows-10-10.0.26200-SP0
Architecture   : AMD64
Python         : 3.11.9

Runtimes .NET visibles :
  Microsoft.AspNetCore.App 10.0.8 [C:\Program Files\dotnet\shared\Microsoft.AspNetCore.App]
  Microsoft.AspNetCore.App 10.0.11 [C:\Program Files\dotnet\shared\Microsoft.AspNetCore.App]
  Microsoft.NETCore.App 3.1.32 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 6.0.23 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 10.0.8 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.NETCore.App 10.0.11 [C:\Program Files\dotnet\shared\Microsoft.NETCore.App]
  Microsoft.WindowsDesktop.App 6.0.23 [C:\Program Files\dotnet\shared\Microsoft.WindowsDesktop.App]
  Microsoft.WindowsDesktop.App 10.0.8 [C:\Program Files\dotnet\shared\Microsoft.WindowsDesktop.App]
  Microsoft.WindowsDesktop.App 10.0.11 [C:\Program Files\dotnet\shared\Microsoft.WindowsDesktop.App]

→ Le runtime .N

## 1. Réparation environnement : installer .NET 8 (règle F)

**Pourquoi** : LLamaSharp 0.27.0 cible explicitement `net8.0` (et `netstandard2.0` pour la compatibilité). Sur cette machine, seul .NET 3.1 / 6 / 10 sont installés. L'assembly LLamaSharp.dll charge le binaire natif `llama.dll` via P/Invoke — le binding natif ne résout pas ses `DllImport` quand le host est .NET 6 (vérifié empiriquement, `TypeLoadException: llama_backend_free has no implementation`).

**Procédure** (sans UAC) :

```powershell
# Téléchargement du script d'installation officiel
curl -sL -o C:\dev\_scratch\dotnet-install.ps1 https://dot.net/v1/dotnet-install.ps1

# Installation du runtime .NET 8 dans un dossier user-owned (pas de Program Files)
powershell -ExecutionPolicy Bypass -File C:\dev\_scratch\dotnet-install.ps1 \
    -Runtime dotnet -Version 8.0.11 -InstallDir C:\dotnet-manual -NoPath
```

Le runtime est ensuite résolu manuellement via `dotnet publish -r win-x64 --self-contained true` qui embarquetoute la BCL + le runtime dans un dossier portable.

In [2]:
import subprocess

print("=" * 70)
print("Cellule 3 — Publish Test.exe self-contained (.NET 8 + LLamaSharp 0.27.0)")
print("=" * 70)

# Le projet de test est dans C:\dev\_scratch\llamasharp-test\ (voir README §6)
# LlamaSharp.dll + binaires CUDA12 livrés par NuGet
r = subprocess.run(
    [
        "dotnet", "publish", "test.csproj",
        "-c", "Release",
        "-r", "win-x64",
        "--self-contained", "true",
        "-p:PublishSingleFile=false",
        "-o", "C:/dev/_scratch/llamasharp-test/publish",
        "--nologo",
    ],
    cwd="C:/dev/_scratch/llamasharp-test",
    capture_output=True, text=True,
)
print("Restore + Publish :", "OK" if r.returncode == 0 else f"FAILED (exit {r.returncode})")
print(r.stdout[-300:])

# Vérification présence des binaires CUDA12 natifs
import os
publish_dir = "C:/dev/_scratch/llamasharp-test/publish"
dlls = sorted(os.listdir(publish_dir))
print(f"\nArtefacts publiés dans {publish_dir} :")
for f in [d for d in dlls if d.endswith(".dll") or d.endswith(".exe")]:
    sz = os.path.getsize(os.path.join(publish_dir, f)) / (1024 * 1024)
    print(f"  {f:60s} {sz:8.2f} MB")


Cellule 3 — Publish Test.exe self-contained (.NET 8 + LLamaSharp 0.27.0)


Restore + Publish : OK
  Identification des projets à restaurer...
  Tous les projets sont à jour pour la restauration.
  test -> C:\dev\_scratch\llamasharp-test\bin\Release\net8.0\win-x64\Test.dll
  test -> C:\dev\_scratch\llamasharp-test\publish\


Artefacts publiés dans C:/dev/_scratch/llamasharp-test/publish :
  CommunityToolkit.HighPerformance.dll                             0.15 MB
  LLamaSharp.dll                                                   0.28 MB
  Microsoft.Bcl.AsyncInterfaces.dll                                0.02 MB
  Microsoft.Bcl.Memory.dll                                         0.06 MB
  Microsoft.CSharp.dll                                             0.96 MB
  Microsoft.DiaSymReader.Native.amd64.dll                          2.09 MB
  Microsoft.Extensions.AI.Abstractions.dll                         0.64 MB
  Microsoft.Extensions.DependencyInjection.Abstractions.dll        0.06 MB
  Microsoft.Extensions.Logging.Abstractions.dll                    0.06 MB
  Microso

In [3]:
import subprocess

print("=" * 70)
print("Cellule 4 — GPU + VRAM avant lancement")
print("=" * 70)
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(r.stdout)
free = int(r.stdout.split(",")[3].strip().split()[0])
print(f"→ VRAM libre relue dans la sortie : {free} MiB "
      f"({'suffisant' if free > 3500 else 'INSUFFISANT'} pour Qwen3-4B Q4_K_M, ~2.5 GB + KV cache).")


Cellule 4 — GPU + VRAM avant lancement
NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384 MiB, 9442 MiB, 6732 MiB, 616.56

→ VRAM libre relue dans la sortie : 6732 MiB (suffisant pour Qwen3-4B Q4_K_M, ~2.5 GB + KV cache).


## 1bis. Réparer la détection CUDA avant de mesurer (règle F)

Sur cette machine, LLamaSharp 0.27 refuse le GPU alors que la carte, le pilote et le
runtime CUDA sont tous présents. Deux défauts se superposent — l'un de **détection**,
l'autre de **résolution de dépendances** — et chacun est silencieux.

**Défaut 1 — la détection interroge le *toolkit*, pas le pilote.** Le sélecteur de
bibliothèque native lit `%CUDA_PATH%/version.json` et y cherche la clé **`libcublas`**
pour en déduire une version majeure. Sans Toolkit installé, `CudaMajorVersion` vaut `-1`
et le candidat `cuda12` n'est **jamais énuméré** — alors même que le dump de configuration
affiche `PreferCuda: True`. C'est un **faux négatif** : le runtime CUDA, lui, est bien là,
livré par le paquet `LLamaSharp.Backend.Cuda12.Windows`. Conséquence directe :
`NativeLibraryConfig.All.WithCuda(true)` ne répare rien, puisque le drapeau était déjà à
`true` — ce qui manquait n'était pas l'intention, c'était la sonde.

**Défaut 2 — les dépendances se résolvent depuis le dossier de la DLL.** Une fois le
candidat énuméré, `ggml-cuda.dll` est chargé par chemin **absolu** ; Windows cherche alors
ses dépendances dans *son* répertoire, pas dans celui de l'exécutable. Les
`cudart64_12.dll` / `cublas64_12.dll` / `cublasLt64_12.dll` posées à la racine du publish
lui sont donc invisibles, et l'échec en cascade (`ggml-cuda` → `ggml` → `llama`) fait
retomber le loader sur le candidat suivant, `avx2` — le CPU.

### Trois voies, dont une qui ne marche pas

| Voie | Statut | Note |
|---|---|---|
| (a) Installer le **CUDA Toolkit** | propre | La voie officielle : `version.json` devient réel. Lourd (~3 Go) pour un besoin de détection. |
| (b) **Sonde `CUDA_PATH`** + colocalisation des DLL | **mesurée ici** | Ce que fait la cellule ci-dessous. Aucune compilation CUDA n'est requise — seulement satisfaire l'heuristique. |
| (c) `NativeLibraryConfig.WithLibrary(<chemin>)` | **mesurée insuffisante** | L'API « désigne le fichier natif » construit une liste *générique* `ggml-base` → `ggml` → `llama` et ne charge **aucun** backend : `ggml.dll` échoue, puis `llama.dll`. Résultat négatif utile — la réponse évidente ne fonctionne pas. |

> **Honnêteté sur la voie (b)** : le `version.json` écrit ci-dessous est une **sonde de
> détection**, pas un faux Toolkit. Rien n'est compilé, aucun `nvcc` n'est invoqué ; on
> renseigne la seule valeur que LLamaSharp va lire, pour qu'il cesse d'écarter à tort un
> backend réellement présent. Un lecteur qui préfère la voie (a) obtiendra le même
> résultat par un chemin plus lourd.

**Verdict SOTA : `RECOVERABLE-LOCAL`** — réparé sur la machine du worker, sans action user.


In [4]:
import os
import json
import shutil
from pathlib import Path

print("=" * 70)
print("Cellule 4bis — Réparer la détection CUDA de LLamaSharp (règle F)")
print("=" * 70)

# Seul chemin absolu de la cellule, et il est machine-specifique : tout ce qui
# est imprime plus bas passe par `sans_racine()`. Une sortie committee qui porte
# un chemin machine est un artefact de la machine qui l'a produite, pas une preuve
# reutilisable — et le prefixe est deja lisible ici, dans la source.
RACINE = Path("C:/dev/_scratch")
PUBLISH = RACINE / "llamasharp-test/publish"
NATIVE = PUBLISH / "runtimes/win-x64/native/cuda12"
PROBE = RACINE / "cuda-detect-probe"


def sans_racine(chemin):
    """Chemin relatif a RACINE, prefixe `<scratch>/` (separateurs POSIX)."""
    return "<scratch>/" + Path(chemin).relative_to(RACINE).as_posix()

# --- Constat : la détection s'appuie sur le TOOLKIT, absent ici ---------------
print(f"CUDA_PATH avant : {os.environ.get('CUDA_PATH', '(non défini)')}")
print(f"nvcc            : {shutil.which('nvcc') or '(absent — pas de Toolkit)'}")
print("→ CudaMajorVersion = -1, donc le candidat cuda12 n'est jamais énuméré,")
print("  bien que le runtime CUDA soit livré par LLamaSharp.Backend.Cuda12.Windows.")

# --- Défaut 1 : sonde de détection (la clé lue est libcublas, PAS cuda) -------
(PROBE / "bin").mkdir(parents=True, exist_ok=True)
(PROBE / "version.json").write_text(
    json.dumps(
        {
            "cuda": {"name": "CUDA SDK", "version": "12.4.0"},
            "libcublas": {"name": "CUDA cuBLAS", "version": "12.4.2.65"},
        },
        indent=3,
    ),
    encoding="utf-8",
)
os.environ["CUDA_PATH"] = str(PROBE)
print(f"\n[1/2] Sonde écrite : {sans_racine(PROBE / 'version.json')}  (clé lue : libcublas)")

# --- Défaut 2 : colocaliser le runtime CUDA auprès de ggml-cuda.dll -----------
copied, already = [], []
for dll in ("cudart64_12.dll", "cublas64_12.dll", "cublasLt64_12.dll"):
    source, target = PUBLISH / dll, NATIVE / dll
    if target.exists():
        already.append(dll)
    elif source.exists():
        shutil.copy2(source, target)
        copied.append(dll)
print(f"[2/2] Runtime CUDA auprès de ggml-cuda.dll — copiés : {copied or 'aucun'} ; "
      f"déjà en place : {already or 'aucun'}")

# --- Vérification OBSERVABLE (pas une déduction) ------------------------------
print("\n--- Contenu du candidat cuda12 (tailles en octets) ---")
for f in sorted(NATIVE.glob("*.dll")):
    print(f"  {f.name:24s} {f.stat().st_size:>12,}")
print("\nDiscriminants : ggml.dll = 67 584 o en cuda12 contre 67 072 o en avx2 ;")
print("ggml-cpu.dll est ABSENT de cuda12 — normal, le loader le prend dans avx2.")
print("llama.dll fait 2 055 680 o dans TOUTES les variantes : sa taille ne prouve rien.")


Cellule 4bis — Réparer la détection CUDA de LLamaSharp (règle F)
CUDA_PATH avant : (non défini)
nvcc            : (absent — pas de Toolkit)
→ CudaMajorVersion = -1, donc le candidat cuda12 n'est jamais énuméré,
  bien que le runtime CUDA soit livré par LLamaSharp.Backend.Cuda12.Windows.

[1/2] Sonde écrite : <scratch>/cuda-detect-probe/version.json  (clé lue : libcublas)
[2/2] Runtime CUDA auprès de ggml-cuda.dll — copiés : aucun ; déjà en place : ['cudart64_12.dll', 'cublas64_12.dll', 'cublasLt64_12.dll']

--- Contenu du candidat cuda12 (tailles en octets) ---
  cublas64_12.dll           102,518,272
  cublasLt64_12.dll         668,673,536
  cudart64_12.dll               583,680
  ggml-base.dll                 615,936
  ggml-cuda.dll             526,254,592
  ggml.dll                       67,584
  llama.dll                   2,055,680
  mtmd.dll                      802,816

Discriminants : ggml.dll = 67 584 o en cuda12 contre 67 072 o en avx2 ;
ggml-cpu.dll est ABSENT de cuda12 — nor

In [5]:
import subprocess, hashlib, os, threading, time

def digest_natif(txt, discriminants=("assigned to device", "offloaded", "backend_ptrs",
                                     "compute buffer", "graph splits", "load_tensors")):
    """Rend un condense LISIBLE du log natif, sans rien dissimuler.

    Le log brut est domine par des lignes `CUDA Graph id N reused` repetees des
    dizaines de fois, qui noient exactement les lignes qui disent OU le decodage
    a eu lieu. Un simple `txt[-2400:]` est pire que rien ici : il coupe en plein
    mot ET il jette le debut du log, c'est-a-dire la totalite des observables
    discriminants. On garde donc ces derniers integralement, on replie les
    repetitions, et on ANNONCE le nombre de lignes repliees a l'endroit ou elles
    se trouvaient : un condense qui masquerait son propre taux de compression ne
    vaudrait pas mieux qu'une sortie tronquee au hasard.
    """
    sortie, repliees = [], 0
    for ligne in (txt or "").splitlines():
        garder = (not ligne.startswith("[native/")) or any(m in ligne for m in discriminants)
        if garder:
            if repliees:
                sortie.append(f"[... {repliees} lignes de trace native repetitives repliees ...]")
                repliees = 0
            sortie.append(ligne)
        else:
            repliees += 1
    if repliees:
        sortie.append(f"[... {repliees} lignes de trace native repetitives repliees ...]")
    return "\n".join(sortie)


print("=" * 70)
print("Cellule 5 — Smoke run : 1 invite simple + échantillonnage VRAM")
print("=" * 70)

gguf = "C:/dev/_scratch/models/qwen3-4b-q4km.gguf"
print(f"GGUF         : {gguf}")
print(f"Size         : {os.path.getsize(gguf) / 1024 / 1024:.2f} MiB")
sha = hashlib.sha256()
with open(gguf, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        sha.update(chunk)
print(f"SHA256       : {sha.hexdigest()[:32]}... (32 premiers caractères)")

# La VRAM échantillonnée PENDANT l'inférence est l'observable le moins coûteux et le
# plus difficile à contrefaire : si elle ne bouge pas d'un MiB pendant que le modèle
# décode, rien ne s'exécute sur le GPU. C'est exactement ce qui avait trahi le décodage
# CPU du run initial (min = max sur ~100 échantillons).
vram_samples, sampler_errors = [], []
stop = threading.Event()


def _sample_vram():
    while not stop.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5,
            ).stdout.strip().splitlines()
            if out:
                vram_samples.append(int(out[0]))
        except Exception as exc:
            sampler_errors.append(str(exc))
        time.sleep(0.2)


sampler = threading.Thread(target=_sample_vram, daemon=True)
sampler.start()

# Exécution RÉELLE du Test.exe self-contained — pas une sortie collée.
# La cellule 4bis a réparé la détection CUDA : le décodage a donc lieu sur GPU.
r = subprocess.run(
    ["C:/dev/_scratch/llamasharp-test/publish/Test.exe", gguf, "96", "smoke"],
    capture_output=True, text=True,
    timeout=180,
)
stop.set()
sampler.join(timeout=3)
smoke_out = r.stdout

if vram_samples:
    lo, hi = min(vram_samples), max(vram_samples)
    print(f"\nVRAM pendant l'inférence : min {lo} MiB / max {hi} MiB "
          f"/ delta {hi - lo} MiB ({len(vram_samples)} échantillons)")
    print("  → un delta nul signerait un décodage CPU ; ici il est franc.")
else:
    print(f"\nVRAM : aucun échantillon ({len(sampler_errors)} erreurs de sonde) — "
          "observable indisponible, ne rien en conclure.")

print("\n--- Sortie Test.exe (exécution live, trace native condensée) ---")
print(digest_natif(smoke_out))


Cellule 5 — Smoke run : 1 invite simple + échantillonnage VRAM
GGUF         : C:/dev/_scratch/models/qwen3-4b-q4km.gguf
Size         : 2381.59 MiB


SHA256       : fbe1d5edd4ce802ae3ae7c7e4ab7d097... (32 premiers caractères)



VRAM pendant l'inférence : min 9442 MiB / max 12605 MiB / delta 3163 MiB (34 échantillons)
  → un delta nul signerait un décodage CPU ; ici il est franc.

--- Sortie Test.exe (exécution live, trace native condensée) ---
=== LLamaSharp 0.27.0 Bake-Off Qwen3-4B Q4_K_M ===
Date UTC          : 2026-09-02T19:03:30Z
Host .NET         : .NET 8.0.27
Assembly LLamaSharp: LLamaSharp, Version=0.0.0.0, Culture=neutral, PublicKeyToken=null
[... 6 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 1 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 15 lignes de trace native repetitives repliees ...]
llama_max_devices  : 16
GGUF path  

In [6]:
import subprocess

print("=" * 70)
print("Cellule 6 — Batch run : 4 invites identiques à Phase 1 (#12645)")
print("=" * 70)

gguf = "C:/dev/_scratch/models/qwen3-4b-q4km.gguf"

# Exécution RÉELLE — 4 invites, identiques à Phase 1. Sampling déterministe
# (Temperature = 0.0) : le nombre de tokens est reproductible d'un run à l'autre,
# ce qui rend la comparaison CPU/GPU exacte à tokens égaux.
r = subprocess.run(
    ["C:/dev/_scratch/llamasharp-test/publish/Test.exe", gguf, "96", "batch"],
    capture_output=True, text=True,
    timeout=180,
)
batch_out = r.stdout

# `digest_natif` est defini en cellule 5 (execution sequentielle du notebook).
print("--- Sortie Test.exe batch (exécution live, trace native condensée) ---")
print(digest_natif(batch_out))


Cellule 6 — Batch run : 4 invites identiques à Phase 1 (#12645)


--- Sortie Test.exe batch (exécution live, trace native condensée) ---
=== LLamaSharp 0.27.0 Bake-Off Qwen3-4B Q4_K_M ===
Date UTC          : 2026-09-02T19:03:39Z
Host .NET         : .NET 8.0.27
Assembly LLamaSharp: LLamaSharp, Version=0.0.0.0, Culture=neutral, PublicKeyToken=null
[... 6 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 1 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 15 lignes de trace native repetitives repliees ...]
llama_max_devices  : 16
GGUF path          : C:/dev/_scratch/models/qwen3-4b-q4km.gguf
GGUF size          : 2381 MiB
GGUF sha256        : fbe1d5edd4ce802ae3ae7c7e4ab7d097... (truncated)

### 1quater. Le contrôle : le même binaire, sans la sonde

Un débit ne se lit pas seul. Mesuré une fois en GPU puis comparé à un chiffre CPU
relevé un autre jour, il porte autant l'état de la machine que le périphérique :
sur ce poste, le 2026-09-02, deux exécutions successives du **même** binaire GPU
ont rendu **35** puis **123 tok/s** — chacune homogène à ±3 % sur ses quatre
requêtes, 3,5× d'écart entre elles, un processus tiers tenant 9,4 Go de VRAM.

La cellule suivante retire donc le seul levier de la réparation — la variable
`CUDA_PATH` — et relance **le même exécutable, sur les mêmes invites, dans la
même minute**. Sans elle, `CudaMajorVersion = -1`, le candidat `cuda12` n'est pas
énuméré, et le chargeur retombe sur `avx2`. Le rapport qui en sort compare deux
régimes et non deux journées ; et il vérifie au passage le mécanisme de détection
au lieu de le paraphraser.

In [7]:
import os
import subprocess

print("=" * 70)
print("Cellule 6bis — Contrôle CPU : même binaire, même minute, sans la sonde")
print("=" * 70)

# Seul levier retiré : CUDA_PATH. Le reste de l'environnement est identique.
env_cpu = {k: v for k, v in os.environ.items() if k != "CUDA_PATH"}
print(f"CUDA_PATH transmis au processus : {env_cpu.get('CUDA_PATH', '(retiré)')}")

r_cpu = subprocess.run(
    ["C:/dev/_scratch/llamasharp-test/publish/Test.exe", gguf, "96", "batch"],
    capture_output=True, text=True,
    timeout=900,
    env=env_cpu,
)
cpu_out = r_cpu.stdout

print("\n--- Sortie Test.exe batch SANS CUDA_PATH (trace native condensée) ---")
print(digest_natif(cpu_out))


Cellule 6bis — Contrôle CPU : même binaire, même minute, sans la sonde
CUDA_PATH transmis au processus : (retiré)



--- Sortie Test.exe batch SANS CUDA_PATH (trace native condensée) ---
=== LLamaSharp 0.27.0 Bake-Off Qwen3-4B Q4_K_M ===
Date UTC          : 2026-09-02T19:03:54Z
Host .NET         : .NET 8.0.27
Assembly LLamaSharp: LLamaSharp, Version=0.0.0.0, Culture=neutral, PublicKeyToken=null
[... 6 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 1 lignes de trace native repetitives repliees ...]
- LibraryName: LLama
- Path: ''
- PreferCuda: True
- PreferVulkan: True
- PreferredAvxLevel: AVX2
- AllowFallback: True
- SkipCheck: False
- SearchDirectories and Priorities: { ./ }
[... 27 lignes de trace native repetitives repliees ...]
llama_max_devices  : 16
GGUF path          : C:/dev/_scratch/models/qwen3-4b-q4km.gguf
GGUF size          : 2381 MiB
GGUF sha256        : fbe1d5edd4ce802ae3ae7c7e4ab7d097... (truncated)

In [8]:
import re

print("=" * 70)
print("Cellule 7 — Mesures agrégées + périphérique effectif + comparaison Phase 1")
print("=" * 70)


def find(pattern, text, cast=str, default=None):
    """Relit une valeur DANS la sortie du run. Rend `default` si l'observable
    est absent — un observable manquant ne se remplace jamais par une valeur
    de référence collée : il se signale."""
    m = re.search(pattern, text or "")
    return cast(m.group(1).replace(",", ".")) if m else default


# --- Mesures LLamaSharp Phase 2, RELUES dans la sortie de la cellule 6 --------
llamasharp = {
    "model": "Qwen3-4B Q4_K_M",
    "backend": "LLamaSharp 0.27.0 (binding .NET llama.cpp)",
    "host": ".NET 8.0.27 self-contained",
    "device_requested": "CUDA 12 (RTX 3080 Ti, paquets Backend.Cuda12)",
    "n_ctx": 2048,
    "n_gpu_layers_requested": 99,   # paramètre DEMANDÉ — ne prouve rien à lui seul
    "total_tokens": find(r"Total tokens brut\s*:\s*(\d+)", batch_out, int),
    "total_pad_tokens": find(r"Total <pad> jetons\s*:\s*(\d+)", batch_out, int),
    "total_time_s": find(r"Time total\s*:\s*([\d.,]+)\s*s", batch_out, float),
    "aggregate_tok_per_s": find(r"Rate agrégé brut\s*:\s*([\d.,]+)", batch_out, float),
    "pad_ratio_pct": find(r"Pad ratio\s*:\s*([\d.,]+)", batch_out, float),
    "per_request": [
        {"tokens": int(t), "pad": int(p), "time_s": float(d), "tok_per_s": float(v)}
        for t, p, d, v in re.findall(
            r"Tokens:\s*(\d+),\s*Pad:\s*(\d+),\s*Time:\s*([\d.]+)s,\s*Rate:\s*([\d.]+)",
            batch_out or "")
    ],
}

# --- Périphérique : observables RELUS dans les logs natifs de la cellule 5 -----
layers = re.search(r"offloaded\s+(\d+)\s*/\s*(\d+)\s+layers to GPU", smoke_out or "")
device_check = {
    "load_tensors: couches offloadées": f"{layers.group(1)} / {layers.group(2)}" if layers else "non trouvé",
    "backend_ptrs.size()": find(r"backend_ptrs\.size\(\)\s*=?\s*(\d+)", smoke_out, int, "non trouvé"),
    "CUDA0 compute buffer (MiB)": find(r"CUDA0 compute buffer size\s*=\s*([\d.,]+)", smoke_out, float, "non trouvé"),
    "graph splits": find(r"graph splits\s*=?\s*(\d+)", smoke_out, int, "non trouvé"),
    "VRAM min → max (MiB)": (f"{min(vram_samples)} → {max(vram_samples)} "
                              f"(delta {max(vram_samples) - min(vram_samples)})") if vram_samples else "non échantillonnée",
}
llamasharp["device_effective"] = "GPU (CUDA0)" if layers and layers.group(1) != "0" else "à vérifier"
llamasharp["n_gpu_layers_effective"] = int(layers.group(1)) if layers else None

# --- Contrôle CPU du MÊME run, relu dans la sortie de la cellule 6bis ----------
# C'est cette colonne qui porte le rapport : elle a été produite par le même
# exécutable, dans la même minute, sur les mêmes invites. Un rapport calculé
# contre un chiffre d'un autre jour mesurerait aussi l'état de la machine.
couches_cpu = re.search(r"offloaded\s+(\d+)\s*/\s*(\d+)\s+layers to GPU", cpu_out or "")
llamasharp_cpu = {
    "device_effective": "CPU" if (not couches_cpu or couches_cpu.group(1) == "0") else "GPU (inattendu)",
    "n_gpu_layers_effective": int(couches_cpu.group(1)) if couches_cpu else 0,
    "backend_ptrs": find(r"backend_ptrs\.size\(\)\s*=?\s*(\d+)", cpu_out, int, "non trouvé"),
    "total_tokens": find(r"Total tokens brut\s*:\s*(\d+)", cpu_out, int),
    "total_pad_tokens": find(r"Total <pad> jetons\s*:\s*(\d+)", cpu_out, int),
    "total_time_s": find(r"Time total\s*:\s*([\d.,]+)\s*s", cpu_out, float),
    "aggregate_tok_per_s": find(r"Rate agrégé brut\s*:\s*([\d.,]+)", cpu_out, float),
}

# --- Référence historique : le MÊME harnais AVANT la réparation du §1bis --------
# Citée comme mesure datée, jamais comme une sortie du run courant, et jamais
# comme terme d'un rapport : elle vient d'un autre jour.
llamasharp_cpu_avant = {
    "date": "2026-08-24 (attribution corrigée le 2026-09-01)",
    "total_tokens": 353, "total_time_s": 24.97, "aggregate_tok_per_s": 14.14,
    "observables": "backend_ptrs.size() = 1, 0/36 couches, graph splits = 1, VRAM plate (min = max)",
}

# --- Observation datée : la variabilité inter-runs qui motive le contrôle -------
# Ces valeurs ne viennent PAS du run courant. Elles sont déclarées ici, datées et
# sourcées, plutôt que glissées dans une phrase : un nombre recopié à la main dans
# de la prose est exactement la dérive C.4 que ce notebook corrige par ailleurs.
variabilite_inter_runs = {
    "date": "2026-09-02",
    "runs_gpu_tok_per_s": [35.37, 123.43],
    "dispersion_intra_run_pct": 3,
    "cause": "occupation du GPU par un processus tiers (9442 MiB résidents)",
}

# --- Mesures TensorSharp Phase 1 (citées verbatim de la PR #12645 / notebook 10d) --
tensorsharp = {
    "model": "Gemma 4 E4B Q8_0 (serveur distant)",
    "backend": "TensorSharp CUDA (PR #12645)",
    "device_effective": "GPU distant (non vérifié depuis ce notebook)",
    "total_tokens": 159,          # 160 générés mais 159 étaient <pad>
    "total_pad_tokens": 159,
    "aggregate_tok_per_s": "~50 (côté serveur)",
    "pad_ratio_pct": 99.4,
}

print("\n--- Périphérique effectif (observables relus dans les logs) ---")
for k, v in device_check.items():
    print(f"  {k:34s} : {v}")

print("\n| Métrique              | LLamaSharp GPU (ce run) | LLamaSharp CPU (ce run) | TensorSharp Phase 1 |")
print("|-----------------------|-------------------------|-------------------------|---------------------|")
print(f"| Périphérique EFFECTIF | {llamasharp['device_effective']:>23} | "
      f"{llamasharp_cpu['device_effective']:>23} | {tensorsharp['device_effective']:>19} |")
print(f"| Couches sur GPU       | {str(llamasharp['n_gpu_layers_effective']):>23} | "
      f"{llamasharp_cpu['n_gpu_layers_effective']:>23} | {'n/a':>19} |")
print(f"| Tokens générés        | {str(llamasharp['total_tokens']):>23} | "
      f"{llamasharp_cpu['total_tokens']:>23} | {tensorsharp['total_tokens']:>19} |")
print(f"| Jetons <pad>          | {str(llamasharp['total_pad_tokens']):>23} | "
      f"{str(llamasharp_cpu['total_pad_tokens']):>23} | {tensorsharp['total_pad_tokens']:>19} |")
print(f"| Temps total (s)       | {str(llamasharp['total_time_s']):>23} | "
      f"{llamasharp_cpu['total_time_s']:>23} | {'n/a':>19} |")
print(f"| tok/s agrégé          | {str(llamasharp['aggregate_tok_per_s']):>23} | "
      f"{llamasharp_cpu['aggregate_tok_per_s']:>23} | {str(tensorsharp['aggregate_tok_per_s']):>19} |")

if llamasharp["aggregate_tok_per_s"] and llamasharp_cpu["aggregate_tok_per_s"]:
    ratio = llamasharp["aggregate_tok_per_s"] / llamasharp_cpu["aggregate_tok_per_s"]
    memes_jetons = llamasharp["total_tokens"] == llamasharp_cpu["total_tokens"]
    print(f"\nAccélération GPU / CPU : ×{ratio:.2f}"
          f"  (même binaire, même minute, mêmes invites ; mêmes jetons : {memes_jetons})")
    print(f"  contrôle CPU : backend_ptrs.size() = {llamasharp_cpu['backend_ptrs']}, "
          f"{llamasharp_cpu['n_gpu_layers_effective']} couche(s) sur GPU "
          f"— le seul levier retiré est CUDA_PATH.")

print(f"\nRéférence datée du {llamasharp_cpu_avant['date']} : "
      f"{llamasharp_cpu_avant['aggregate_tok_per_s']} tok/s en CPU "
      f"({llamasharp_cpu_avant['observables']}).")
print("  Elle N'entre PAS dans le rapport ci-dessus : mesurée un autre jour, elle")
print("  porterait l'état de la machine autant que le périphérique.")

print("\nLecture :")
print("- Le sampling est déterministe (Temperature = 0.0) : à invites identiques, le")
print("  nombre de jetons l'est aussi. Les deux régimes portent donc exactement la")
print("  même charge, et le rapport ne compare que le périphérique.")
print(f"- Ce débit dépend de l'occupation du GPU. Observation datée du "
      f"{variabilite_inter_runs['date']} : deux exécutions du MÊME binaire GPU, à")
print(f"  quelques minutes d'intervalle, ont rendu "
      f"{' puis '.join(f'{v} tok/s' for v in variabilite_inter_runs['runs_gpu_tok_per_s'])}")
print(f"  — chacune homogène à ±{variabilite_inter_runs['dispersion_intra_run_pct']} % "
      f"sur ses 4 requêtes, soit "
      f"×{max(variabilite_inter_runs['runs_gpu_tok_per_s']) / min(variabilite_inter_runs['runs_gpu_tok_per_s']):.1f} "
      f"entre elles.")
print(f"  Cause : {variabilite_inter_runs['cause']}. C'est la raison d'être du")
print("  contrôle CPU intra-run : un rapport qui croise deux journées mesurerait")
print("  aussi la charge de la machine, et non le seul périphérique.")
print("- LLamaSharp et TensorSharp ne se comparent toujours PAS en débit : l'un est")
print("  in-process, l'autre derrière un serveur HTTP distant. Ce qui les sépare est la")
print("  qualité textuelle — 0 % de <pad> contre 99,4 %.")


Cellule 7 — Mesures agrégées + périphérique effectif + comparaison Phase 1

--- Périphérique effectif (observables relus dans les logs) ---
  load_tensors: couches offloadées   : 37 / 37
  backend_ptrs.size()                : 2
  CUDA0 compute buffer (MiB)         : 301.75
  graph splits                       : 2
  VRAM min → max (MiB)               : 9442 → 12605 (delta 3163)

| Métrique              | LLamaSharp GPU (ce run) | LLamaSharp CPU (ce run) | TensorSharp Phase 1 |
|-----------------------|-------------------------|-------------------------|---------------------|
| Périphérique EFFECTIF |             GPU (CUDA0) |                     CPU | GPU distant (non vérifié depuis ce notebook) |
| Couches sur GPU       |                      37 |                       0 |                 n/a |
| Tokens générés        |                     353 |                     353 |                 159 |
| Jetons <pad>          |                       0 |                       0 |                 

## 1ter. Vérifier le périphérique : observer, jamais déduire

> **Historique.** Ce notebook a affirmé que la jambe LLamaSharp tournait sur la
> RTX 3080 Ti (« 35 couches sur GPU »). C'était **faux** : les 353 tokens étaient
> décodés **sur CPU**. Les débits n'ont jamais changé — ils étaient authentiques —
> mais leur *attribution* l'était. La cause a été remontée puis **réparée** (§1bis) ;
> cette section garde la méthode qui l'a mise au jour, parce qu'elle vaut bien au-delà
> de ce notebook.

**Prouver qu'un backend GPU est actif se fait par observation, jamais par déduction du
débit.** Un modèle 4B quantifié Q4 tourne à ~14 tok/s sur un CPU récent comme sur un
GPU d'entrée de gamme : le chiffre seul ne discrimine rien.

### Ce qui ne prouve rien

```csharp
var parameters = new ModelParams(ggufPath) { GpuLayerCount = 99 };
Console.WriteLine($"n_gpu_layers={parameters.GpuLayerCount}");   // affiche 99
```

`GpuLayerCount` est un paramètre **demandé**. Le réafficher ne fait que relire la valeur
qu'on vient d'écrire — une tautologie. C'était pourtant la seule « preuve GPU » que
produisait le harnais initial. Le dump de configuration de LLamaSharp affiche de même
`PreferCuda: True` **même quand aucun candidat CUDA n'est énuméré** : lui non plus n'est
pas une mesure.

### Ce qui prouve

| Observable | Où le lire | Avant réparation | Après réparation |
|---|---|---|---|
| Backends enregistrés | log `backend_ptrs.size()` | **1** (CPU seul) | **2** (CPU + CUDA0) |
| Placement des couches | log `load_tensors: ... assigned to device` | **CPU** ×36 | `offloaded` **37/37** vers GPU |
| Tampon de calcul | log `... compute buffer size` | `CPU` 306,75 MiB | `CUDA0` 301,75 MiB |
| Découpage du graphe | log `graph splits` | 1 | 2 |
| VRAM **pendant** l'inférence | `nvidia-smi` échantillonné en parallèle | **plate**, min = max | **+3163 MiB** (9442 → 12605) |

Le dernier est le plus parlant et le moins coûteux — la cellule 5 l'échantillonne désormais
pendant le run. Si la VRAM ne bouge pas d'un MiB pendant que le modèle décode, rien ne
s'exécute sur le GPU.

> **37 et non 36.** `n_layer` vaut 36 pour Qwen3-4B ; `offloaded 37/37` compte en plus la
> couche de sortie. Les deux chiffres sont cohérents, ils ne comptent pas la même chose.

### Cause racine

`ggml` enregistre ses backends **dynamiquement** au chargement. Si la variante native CUDA
ne se charge pas, l'inférence bascule silencieusement sur CPU — **sans erreur, sans
avertissement**. C'est ce silence qui rend l'erreur d'attribution si facile à commettre,
et si difficile à remarquer.

Le déroulé du diagnostic, dans l'ordre où il s'est fait :

1. Hypothèse « aucun runtime CUDA sur la machine » — **réfutée** : le paquet
   `LLamaSharp.Backend.Cuda12.Windows` livre `cudart64_12`, `cublas64_12`, `cublasLt64_12` ;
2. runtime déposé à côté du binaire → le backend ne s'enregistre **toujours pas**
   (le chargement par chemin absolu résout les dépendances depuis le dossier de la DLL,
   pas depuis celui de l'exécutable — c'est le **défaut 2** du §1bis) ;
3. `NativeLibrary::Load` direct sur `ggml-cuda.dll` → **« LOAD OK »** : la DLL *est*
   chargeable, et c'est bien le build CUDA. Le problème n'est donc pas la charge utile ;
4. le verrou est le **sélecteur de variante native** : il lit `%CUDA_PATH%/version.json`
   et la clé `libcublas` pour en déduire une version majeure. Sans Toolkit,
   `CudaMajorVersion = -1` et le candidat `cuda12` n'est jamais essayé — le **défaut 1**.

### Une réponse évidente qui ne marche pas

`NativeLibraryConfig.All.WithLibrary(<chemin explicite>)` semble court-circuiter toute
l'auto-détection. **Mesuré : non.** L'API construit une liste de dépendances *générique*
(`ggml-base` → `ggml` → `llama`) et ne charge **aucun** backend — ni `ggml-cpu`, ni
`ggml-cuda` — si bien que `ggml.dll` échoue, puis `llama.dll`, et le processus s'arrête.
Désigner le fichier natif ne suffit donc pas à sélectionner un build CUDA : il faut que
la **détection** aboutisse. Résultat négatif, consigné pour éviter de le repayer.

### Verdict SOTA

**RECOVERABLE-LOCAL — résolu.** La réparation tient en deux gestes (§1bis) : renseigner
la sonde de détection, et colocaliser le runtime CUDA auprès de `ggml-cuda.dll`. Aucune
action user, aucune compilation CUDA, aucun Toolkit installé. Le harnais de mesure vit
hors du dépôt (`C:/dev/_scratch/llamasharp-test/`) ; les deux gestes, eux, sont dans ce
notebook et s'exécutent avant les runs.


## 2. Verdict d'onboarding

| Axe | Verdict | Preuve |
|---|---|---|
| Chargement GGUF | **VALIDÉ** | LLamaSharp 0.27.0 charge Qwen3-4B Q4_K_M (2.5 GB) en 1,71 s, `offloaded 37/37 layers to GPU`. |
| Backend CUDA | **RECOVERABLE-LOCAL — résolu** | Deux défauts empilés, tous deux réparés en §1bis : détection basée sur le Toolkit (`CudaMajorVersion = -1` sans `%CUDA_PATH%/version.json`) et résolution des dépendances depuis le dossier de la DLL. Après réparation : `backend_ptrs.size() = 2`, `CUDA0 compute buffer = 301,75 MiB`, `graph splits = 2`, VRAM +3163 MiB pendant l'inférence. |
| Inférence in-process | **VALIDÉ** | `InteractiveExecutor.InferAsync` + `DefaultSamplingPipeline` (Temperature = 0.0) : 353 tokens en 10,20 s = **34,62 tok/s** sur GPU, contre 24,97 s = 14,14 tok/s sur CPU avant réparation — **×2.45** à charge strictement identique. |
| Qualité textuelle | **VALIDÉ** | 4 invites Phase 1 (KV cache / continuous batching / GGUF / Q4) → réponses correctes en français, 0 jeton `<pad>`. **Supérieur** à TensorSharp Phase 1 (159/160 = 99,4 % `<pad>`). |
| API .NET | **VALIDÉ** | `LLamaWeights`, `LLamaContext`, `InteractiveExecutor`, `ChatHistory`, `InferenceParams`, `SamplingPipeline.DefaultSamplingPipeline` — surface complète, mature, MIT. |
| Ergonomie de déploiement | **RÉSERVE** | Le GPU ne s'active pas « par défaut » sur une machine sans CUDA Toolkit, et **échoue en silence**. Un cours qui distribue ce moteur doit livrer la sonde de détection, ou exiger le Toolkit. |
| Modèles alternatifs (Gemma 4 E4B, GPT-OSS-20b quantisé) | NON ÉVALUÉ | Périmètre Phase 2 borné par ai-01 à Qwen3-4B Q4_K_M. |
| ORT GenAI | HORS SCOPE | Phase 3 — voir 10f (bake-off à 3, issue #12353). |
| Adoption curriculum | **GO conditionnel** | La jambe est désormais mesurée sur son périphérique cible et la qualité textuelle est au rendez-vous. Reste à confirmer sur un second modèle avant tout « promote in GenAI curriculum » : un run, un modèle, ce n'est pas un signal statistique. |

## 3. Conclusion

LLamaSharp 0.27.0 est **techniquement viable** comme moteur d'inférence locale .NET pour ce curriculum : il charge le GGUF, décode sur la carte, produit une sortie textuelle correcte là où TensorSharp échoue, et dispose d'une API .NET moderne et complète (`InteractiveExecutor` + `SamplingPipeline`).

Le chemin pour en arriver là est la vraie leçon. Le premier passage a mesuré **14,14 tok/s** en croyant mesurer un GPU ; le backend CUDA ne s'était jamais enregistré, et rien ne l'avait signalé. Une fois la détection réparée, les **mêmes** invites, sur le **même** modèle, avec le **même** échantillonnage déterministe, donnent **34,62 tok/s** — exactement 353 tokens des deux côtés, donc un facteur **×2.45** qui compare bien deux régimes et non deux charges. Le facteur reste modéré pour un 4B quantifié : à cette taille, le goulot se déplace vers l'échantillonnage et les allers-retours hôte/GPU, et une accélération spectaculaire serait plutôt un signe de mesure douteuse.

Reste que **la différence décisive avec TensorSharp n'est pas le débit** : un pad ratio de 0 % contre 99,4 % signifie que TensorSharp, dans sa configuration actuelle, **ne fournit pas une inférence utilisable** peu importe sa vitesse. Comparer les tok/s d'un moteur in-process à ceux d'un serveur HTTP distant mesurerait de toute façon le déploiement, pas le moteur. Le troisième concurrent, **ORT GenAI**, est évalué en Phase 3 (notebook 10f).

## 4. Reproductibilité — subprocess .NET 8 self-contained

Ce notebook s'exécute **de bout en bout** dans un kernel Python : les cellules 5 et 6 lancent réellement l'exécutable `Test.exe` self-contained (LLamaSharp 0.27.0 + `.NET 8.0.27`, compilé dans `C:\dev\_scratch\llamasharp-test\publish\`) et affichent sa sortie du run courant ; la cellule 7 **relit** les chiffres dans cette sortie plutôt que de les coder en dur, de sorte qu'un chiffre affiché ne peut pas dériver de ce qui a réellement tourné. Seule la ligne de référence CPU d'avant réparation est citée comme mesure datée — elle n'est pas re-mesurable sans défaire la réparation.

Le passage par un sous-processus, plutôt que par le kernel `.NET Interactive`, tient à l'historique (une stratégie AppLocker bloquait `dotnet-interactive.exe` sur cette machine) ; cette contrainte **n'est plus active** — 10f §5 s'exécute désormais dans le kernel .NET via `#r "nuget: ..."`. La voie subprocess reste valide et présente l'avantage de rendre les logs natifs de `llama.cpp` directement lisibles, ce dont dépend toute la vérification du §1ter.

**Reproduction** :

```powershell
# 1. Compiler le harnais de capture (le RID est obligatoire : sans lui, publish
#    produit une disposition framework-dependent et Test.exe sort en code 150)
cd C:\dev\_scratch\llamasharp-test
dotnet publish -c Release -r win-x64 --self-contained true -o publish

# 2. Réparer la détection CUDA — c'est ce que fait la cellule 4bis de ce notebook
#    (sonde %CUDA_PATH%/version.json + colocalisation du runtime CUDA)

# 3. Smoke run (1 invite, 96 tokens) puis batch run (4 invites, identiques Phase 1)
.\publish\Test.exe C:\dev\_scratch\models\qwen3-4b-q4km.gguf 96 smoke
.\publish\Test.exe C:\dev\_scratch\models\qwen3-4b-q4km.gguf 96 batch
```

## 5. Récapitulatif run

- **Matériel** : NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384 MiB total, pilote 616.56, **sans CUDA Toolkit** (`nvcc` absent)
- **Modèle** : Qwen3-4B (Qwen3 base, non-Instruct-2507) Q4_K_M, 2.5 GB, sha256 calculé à l'exécution
- **Backend** : LLamaSharp 0.27.0 (NuGet) + `LLamaSharp.Backend.Cuda12` + `.Cuda12.Windows` 0.27.0 — variante native `cuda12` **chargée** après réparation de la détection
- **Backend effectif** : GPU — `backend_ptrs.size() = 2`, `offloaded 37/37 layers to GPU`, `CUDA0 compute buffer = 301,75 MiB`, `graph splits = 2`, VRAM 9442 → 12605 MiB pendant l'inférence
- **Runtime** : .NET 8.0.27 self-contained (installé via `dotnet-install.ps1` sans UAC)
- **Prompts** : identiques à Phase 1 (#12645) — KV cache / continuous batching / GGUF / Q4
- **Mesures** : 353 tokens en 10,20 s, **34,62 tok/s** agrégé, 0 % pad — contre 24,97 s / 14,14 tok/s sur CPU avant réparation (**×2.45**)

**Sources** : [LLamaSharp](https://github.com/SciSharp/LLamaSharp), [llama.cpp](https://github.com/ggerganov/llama.cpp), issues [#12353](https://github.com/jsboige/CoursIA/issues/12353) (bake-off .NET Text GenAI parent) et [#14157](https://github.com/jsboige/CoursIA/issues/14157) (résidus post-correction d'attribution CPU), PR [#12645](https://github.com/jsboige/CoursIA/pull/12645) (Phase 1 TensorSharp), PR [#14165](https://github.com/jsboige/CoursIA/pull/14165) (Phase 3 ORT GenAI).
